# SLaM Recipe — student walkthrough

## Learning to Autolens / Examples / compound_lens_zoo

---

**What SLaM is.** A canonical PyAutoLens 4-stage pipeline that decomposes a single complex fit into 4 sequential searches with prior passing between them:

1. **SOURCE LP** — initial source as parametric `Sersic`-like profile. Mass model: simple `Isothermal`. Goal: find the right basin for the mass-source posterior.
2. **SOURCE PIX** — promote source to pixelized (`Hilbert` mesh + adaptive regularization). Mass priors fixed from Stage 1. Goal: capture asymmetric / complex source structure that a parametric source can't.
3. **LIGHT LP** — re-fit lens light using the converged source from Stage 2 to mask out the lensed flux. MGE basis. Goal: clean light subtraction for use in Stage 4.
4. **MASS TOTAL** — final mass fit (typically `PowerLaw`) with light + source priors-passed from Stages 2-3. Goal: best-quality mass model.

**Why staged.** A single search across all four sub-problems would have ~50+ free parameters in a degenerate landscape and not converge. Each stage isolates one sub-problem at ~10-20 free params, which Nautilus handles cleanly.

**This recipe** demonstrates:
1. The autolens 2026.4 SLaM API surface (`af.Model.take_attributes`, `result.model.galaxies.x`).
2. Where to find the canonical SLaM source (`autolens_workspace_latest/scripts/guides/modeling/slam_start_here.py`).
3. The minimum 3-stage variant (skipping pixelization) we use in `group_scale/fit_example_group_scale_slam.py`.
4. Common gotchas (stage cascade, prior misalignment, MGE coupling).

---

## Step 1 — the SLaM `Result` API

**The single most important pattern.** PyAutoFit's `Result` object exposes the converged search as `result.instance` (a frozen instance with point values) AND as `result.model` (a `af.Model` with priors centred on the posterior). The `result.model` form is what you pass forward to the next stage:

```python
# Stage 1 result
result_1 = source_lp_search.fit(...)

# In Stage 2, pass the lens mass posterior as priors:
lens_2 = af.Model(
    al.Galaxy, redshift=0.5,
    mass=result_1.model.galaxies.lens.mass,        # ← this is the magic
    bulge=result_1.model.galaxies.lens.bulge,      # ← same for light
)
```

Behind the scenes: `result_1.model.galaxies.lens.mass` is an `af.Model(Isothermal)` with each parameter set to a `GaussianPrior` centred on its posterior median, with sigma equal to the posterior standard deviation. Tight enough to anchor, wide enough to refine in the next stage.

In [ ]:
import os
os.environ.setdefault("PYAUTOFIT_TEST_MODE", "1")
from pathlib import Path
import autofit as af
import autolens as al
from IPython.display import Markdown, display
%matplotlib inline

# Show the model construction patterns. Replace `result_1` with a real Result
# in production.
print("SLaM patterns:")
print("  result.instance.galaxies.lens     # frozen instance — for fixed-priors usage")
print("  result.model.galaxies.lens.mass    # Model with GaussianPrior on each param")
print("  af.Model.take_attributes(source=result.model)  # copy attrs, helper for chained models")

## Step 2 — the canonical 4-stage SLaM

The canonical SLaM pipeline lives at `autolens_workspace_latest/scripts/guides/modeling/slam_start_here.py`. Read this first if you're building a new SLaM driver. The function signatures it uses:

```python
from slam import source_lp
from slam import source_pix
from slam import light_lp
from slam import mass_total

result_1 = source_lp.run(
    settings_search=settings,
    analysis=analysis,
    lens_bulge=al.lp.Sersic,
    mass=al.mp.Isothermal,
    source_bulge=al.lp.SersicCore,
)
result_2 = source_pix.run(
    settings_search=settings,
    analysis=analysis,
    source_lp_results=result_1,        # passes lens mass as priors
    pixelization=al.Pixelization(...),
)
result_3 = light_lp.run(
    settings_search=settings,
    analysis=analysis,
    source_pix_results=result_2,        # mass + source fixed, light free
    lens_bulge=al.lp.Basis,             # MGE
)
result_4 = mass_total.run(
    settings_search=settings,
    analysis=analysis,
    source_results=result_2,
    light_results=result_3,
    mass=al.mp.PowerLaw,                # final mass
)
```

Inside each `slam.<stage>.run()` is the prior-passing logic that this recipe demonstrates manually.

## Step 3 — the minimum 3-stage variant

Skip the SOURCE PIX stage when:
- The source is parametric (Sersic) and well-fit by SOURCE LP.
- You're on a fast budget and don't need the pixelized residual map.
- The data is single-band, low S/N — pixelization will over-fit noise.

The 3-stage variant we ship for `group_scale` (`Modules/10_Cluster_Computing/scripts/fit_example_group_scale_slam.py`):

1. **`slam_source_lp_0`** — light only (MGE bulges for every galaxy, no mass, no source). Subtracts lens light cleanly.
2. **`slam_source_lp_1`** — light fixed, parametric MGE source + Iso mass on every galaxy added.
3. **`slam_mass_total`** — light + source fixed, mass refined to PowerLaw.

The 3-stage variant is the *minimum viable SLaM* — it teaches the staged pattern without the pixelization complexity. Once you understand it, the canonical 4-stage adds source_pix between `source_lp_1` and `light_lp`.

## Step 4 — common gotchas

**The SLaM Stage Cascade.** When an early stage fails (low log_Z, poor residuals), every downstream stage inherits its broken priors and lands in the same wrong basin. The diagnosis is always *open the FIRST stage's `fit_subplot.png`* before chasing problems in later stages. If Stage 1 is bad, fixing Stage 4 won't help.

**Prior-passing footguns.**
1. **Mass profile change between stages.** If Stage 1 used `Isothermal` and Stage 2 promotes to `PowerLaw`, you can't `take_attributes(source=stage_1.model)` — the parameter sets are different. Use `mass = af.Model(al.mp.PowerLaw); mass.einstein_radius = stage_1.model.galaxies.lens.mass.einstein_radius` (per-attribute).
2. **Galaxy collection key mismatch.** If Stage 1 had `lens_0` + `lens_1` and Stage 2 has `lens` + `extra_galaxies`, the prior-passing fails silently. Keep galaxy keys consistent across stages.
3. **Fixed light, free centre.** When passing the converged light to the next stage, you usually want both light *and* its centre fixed — otherwise the chain re-explores light orientation pointlessly.

**MGE coupling between stages.** If Stage 1 used `gaussian_per_basis=2` (bulge+envelope), Stage 2 inherits that structure. Don't change `gaussian_per_basis` between stages; you'll lose the prior-passing alignment.

**Wall time.** A 4-stage SLaM with pixelization is **2-3× the cost of a single direct fit** that uses the same final mass. The wins: it actually converges, and the result is publication-grade. The losses: complexity + debugging cost across 4 sequential failures.

## Step 5 — the converged reference

The compound_lens example has the most-tested SLaM result in this repo. `Examples/compound_lens/02_compound_slam.ipynb` walks through the staged-vs-effective comparison; the converged 5-stage chain landed at `Examples/compound_lens/results/slam_staged/`.

For group-scale-with-MGE, the 3-stage variant lives at `Examples/group_scale/results/slam_*/` (Cannon job 9585470, 8h 33m wall time on 1-BGG + 3-satellites + 1-source — note: that fit converged structurally but failed `/autolens-fit-diagnostics` on residuals, documented as a learning case).

For compound_lens_zoo we currently use the simpler `R5_staged` pattern (a 2-stage chain: R2_2src → R5 with prior passing) rather than the full 4-stage SLaM, because the 5-mock zoo prioritises throughput over per-mock optimisation.

---

## See also

- [`02_compound_lens_ladder.ipynb`](02_compound_lens_ladder.ipynb) — the R0→R5 ladder + R5_staged (compound zoo's 2-stage SLaM variant)
- [`../compound_lens/02_compound_slam.ipynb`](../compound_lens/02_compound_slam.ipynb) — full 5-stage SLaM converged reference
- [`../../Modules/05_Pixelized_Source_Reconstructions/06_pixelization_recipe.ipynb`](../../Modules/05_Pixelized_Source_Reconstructions/06_pixelization_recipe.ipynb) — sister recipe for the SOURCE PIX stage
- [`../../Modules/09_MGE_Linear_Light_Profiles/05_mge_recipe.ipynb`](../../Modules/09_MGE_Linear_Light_Profiles/05_mge_recipe.ipynb) — sister recipe for the LIGHT LP stage's MGE
- `autolens_workspace_latest/scripts/guides/modeling/slam_start_here.py` — the canonical 4-stage SLaM source
- `autolens_workspace_latest/scripts/group/slam.py` — group-scale 6-stage SLaM with extra/scaling galaxies
- `Modules/10_Cluster_Computing/scripts/fit_example_group_scale_slam.py` — our 3-stage minimum-viable adaptation